# Task B -- `D0_V0_noaux`, five seeds, every row, no holdout

The TAPT configuration that led the data-processing grid, rebuilt on 100% of the data
at both stages. Nothing is held out, filtered or deduplicated.

| grid factor | setting here | meaning |
|---|---|---|
| `corpus` | **D0** | TAPT on Kannada text only: `multiclass_train.csv` + OffensEval Kannada |
| `vocab` | **V0** | MuRIL's tokenizer as shipped, no `extend_vocab.py` |
| `aux` | **no** | plain 6-way head, `--aux-weight 0` |

| stage | grid cell `D0_V0_noaux` | `f_tapt` | this notebook |
|---|---|---|---|
| TAPT text | 6,044 of 6,406 | 6,044 of 6,406 | **6,406 of 6,406** |
| classifier rows per model | 2,671 (85%) | 3,143 (deduped) | **3,159 of 3,159** |
| seeds, averaged | 1 | 5 | **5** (42-46) |
| score | 0.5972, 472-row holdout | 0.6007 CodaBench (probably) | CodaBench only |

**There is no local F1, and there cannot be one.** Every labelled row is in training. The
score comes from uploading `d0v0_noaux_full.zip` to the Task B **validation** phase on
CodaBench, which scores it against the 395 gold labels of
`multiclass_validation_inputs.csv`.

About 2 hours on a T4. Upload this file only; it clones `task-b` itself.
Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on.
**Save Version -> Save & Run All.** Do not run it interactively.

In [ ]:
import os, re, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK); os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True); print("cwd:", os.getcwd())
subprocess.run(["git","log","-1","--oneline"], check=True)

# tapt.py gained --val-frac 0 / --min-words / --no-dedupe on task-b. A clone
# without them would silently hold 5% back, so stop instead.
assert "--min-words" in pathlib.Path("src/hastika/task_b/tapt.py").read_text(), \
    "this clone of task-b predates the full-data TAPT flags -- push task-b first"

subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

# ---- the recipe, in one place ------------------------------------------------
D0_CORPUS   = ["data/raw/multiclass_train.csv", "data/external/offenseval_kn.csv"]
V0_MODEL    = "google/muril-base-cased"      # tokenizer as shipped
TAPT_EPOCHS = "8"                            # grid_b.py --tapt-epochs-small
AUX_WEIGHT  = "0"                            # no auxiliary head
SEEDS       = ["42", "43", "44", "45", "46"]
EPOCHS      = "6"
TAPT_OUT    = "artifacts/runs/tapt-d0v0-100"
TAG         = "b_d0v0_noaux_full"
ZIP         = "/kaggle/working/d0v0_noaux_full.zip"

def run(cmd, log=None):
    """Streams, tees, and raises. Nothing here is allowed to fail quietly."""
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")

## 1. TAPT on every D0 comment (~25 min)

`--val-frac 0` holds nothing back, so there is no perplexity line; the epoch lines show
training loss only. `--min-words 1` keeps one-word comments and `--no-dedupe` keeps
repeated ones. The next cell reads the log and stops the notebook if anything was left out.

In [ ]:
run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
     "--model", V0_MODEL, "--corpus", *D0_CORPUS, "--epochs", TAPT_EPOCHS,
     "--val-frac", "0", "--min-words", "1", "--no-dedupe",
     "--out", TAPT_OUT],
    log="artifacts/logs/tapt_d0v0_100.log")

In [ ]:
t = pathlib.Path("artifacts/logs/tapt_d0v0_100.log").read_text()
m = re.search(r"MLM trains on (\d+) of (\d+) comments, (\d+) held out", t)
assert m, "could not find the corpus line in the TAPT log"
used, total, held = map(int, m.groups())
print(f"TAPT trained on {used} of {total} comments, {held} held out")
assert used == total == 6406 and held == 0, "TAPT did not use all 6,406 comments"

## 2. Five seeds on every row, no aux head (~95 min)

`--folds 1` trains each seed on all rows and keeps its final checkpoint, since there is no
validation split to select one with. `--no-dedupe` keeps all 3,159. The five models' test
probabilities are averaged.

In [ ]:
run([sys.executable, "-u", "-m", "hastika.task_b.train", "--tag", TAG,
     "--model", TAPT_OUT, "--folds", "1", "--no-dedupe",
     "--aux-weight", AUX_WEIGHT, "--seeds", *SEEDS, "--epochs", EPOCHS],
    log=f"artifacts/logs/{TAG}.log")

In [ ]:
t = pathlib.Path(f"artifacts/logs/{TAG}.log").read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", t)
for seed, rows in fits:
    print(f"seed {seed}: trained on {rows} rows")
assert [s for s, _ in fits] == SEEDS, "not all five seeds ran"
assert all(r == "3159" for _, r in fits), "a seed trained on fewer than 3,159 rows"

## 3. Package

`agrees with` is how often these 395 predictions match an earlier submission. It is not a
score: it only says whether to expect a similar number. `b_tapt_5f` scored 0.5922 on
CodaBench.

In [ ]:
import numpy as np, pandas as pd
run([sys.executable, "-m", "hastika.common.submission", "--task", "b",
     "--pred", f"artifacts/runs/{TAG}/predictions.csv", "--out", ZIP])
for f in ("artifacts/logs/tapt_d0v0_100.log", f"artifacts/logs/{TAG}.log"):
    run(["cp", f, "/kaggle/working/"])
run(["cp", f"artifacts/runs/{TAG}/predictions.csv", f"/kaggle/working/{TAG}_predictions.csv"])
# 395 x 6 averaged probabilities, kept so later runs can be seed-ensembled with this one
run(["cp", f"artifacts/runs/{TAG}/test_probs.npy", f"/kaggle/working/{TAG}_test_probs.npy"])
run(["unzip", "-l", ZIP])

new = pd.read_csv(f"artifacts/runs/{TAG}/predictions.csv").set_index("id")["label"]
ref = pd.read_csv("submissions/b_tapt_5f/predictions.csv").set_index("id")["label"]
print(f"\nagrees with b_tapt_5f on {100*(new.reindex(ref.index) == ref).mean():.1f}% of test rows")

prior = pd.read_csv("data/raw/multiclass_train.csv")["Hate Category"].value_counts(normalize=True)
dist = pd.DataFrame({"predicted %": 100*new.value_counts(normalize=True),
                     "training %": 100*prior}).fillna(0).round(1)
print("\npredicted distribution vs training prior:")
print(dist.to_string())
print(f"\nlargest drift from the prior: {(dist['predicted %']-dist['training %']).abs().max():.1f} points "
      "(about 3 is normal; a collapsed model reads 50+)")
print("\nDownload d0v0_noaux_full.zip from the Output tab and upload it to the Task B validation phase.")

## 4. After CodaBench scores it

The CodaBench macro-F1 is this run's only score. Record it in `docs/EXPERIMENTS.md` and
`submissions/README.md` next to `b_tapt_5f` (0.5922) and `f_tapt` (0.6007).

395 rows carry about three points of standard deviation, so a gap smaller than that between
two submissions is the test set being small, not one recipe beating the other. Download the
logs, `d0v0_noaux_full.zip` and the `_test_probs.npy` file individually rather than using
Download All: the TAPT checkpoint under `hastika/artifacts/runs/` is about a gigabyte.